# NB04 — Memory-bounded lexical blocking

Stage S4. This notebook writes compact normalized Parquet index parts in chunks, then uses DuckDB to evaluate the union of name and address blocking. It does not create a leaderboard submission.

S3 mined tables are used when available; otherwise the notebook stops rather than silently falling back to an untracked configuration.

In [ ]:
import json, os, pathlib, sys, time
import pandas as pd
ON_SAGEMAKER = pathlib.Path('/opt/ml').exists() or 'SM_CURRENT_HOST' in os.environ
ROOT = pathlib.Path(os.environ.get('BER_ROOT', '/home/ec2-user/SageMaker/ber' if ON_SAGEMAKER else '/Users/pocketfm/Desktop/amazon_ML/Working Folder'))
DATA = ROOT / 'student_resource/dataset'
ART = ROOT / 'artifacts'
ART.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / 'src'))
from business_entity_resolution.blocking_stream import parquet_glob, write_index_parts
from business_entity_resolution.normalization_mined import load_mined_tables
mined_path = ART / 'normalization_v1.json'
if not mined_path.exists():
    raise FileNotFoundError('Run NB03 and create artifacts/normalization_v1.json first')
mined = load_mined_tables(mined_path)
print(f'ROOT={ROOT} mined_config={dict(mined.config)}')

In [ ]:
index_root = ART / 'nb04_index_v1'
t0 = time.time()
counts = {}
for name in ('train_source1', 'train_source2', 'train_source3'):
    rows, parts = write_index_parts(DATA / 'train' / f'{name}.tsv', index_root / name, mined=mined, chunksize=100_000)
    counts[name] = {'rows': rows, 'parts': parts}
    print(f'{name}: rows={rows:,} parts={parts} elapsed={time.time()-t0:.1f}s')
(ART / 'nb04_index_v1_summary.json').write_text(json.dumps({'stage': 'S4', 'counts': counts, 'config': dict(mined.config)}, indent=2) + '\n')

In [ ]:
import duckdb
con = duckdb.connect(database=str(ART / 'nb04.duckdb'))
con.execute("PRAGMA threads=2")
con.execute("PRAGMA memory_limit='5GB'")
(ART / 'nb04_tmp').mkdir(parents=True, exist_ok=True)
con.execute("SET temp_directory = '" + str(ART / 'nb04_tmp').replace('\\', '\\\\').replace("'", "''") + "'")
s1 = parquet_glob(index_root / 'train_source1')
s2 = parquet_glob(index_root / 'train_source2')
s3 = parquet_glob(index_root / 'train_source3')
holdout = ART / 'holdout_s1.parquet'
con.execute("CREATE OR REPLACE TEMP TABLE holdout_ids AS SELECT entity_id FROM read_parquet(?)", [str(holdout)])
query = f"""
WITH refs AS (SELECT * FROM read_parquet('{s1}') WHERE entity_id IN (SELECT entity_id FROM holdout_ids)),
all_candidates AS (
  SELECT r.entity_id AS source1_entity_id, x.entity_id AS candidate_id, 'name' AS block
  FROM refs r JOIN (SELECT * FROM read_parquet('{s2}') UNION ALL SELECT * FROM read_parquet('{s3}')) x
    ON r.country=x.country AND r.name_key=x.name_key AND r.name_key <> ''
  UNION
  SELECT r.entity_id, x.entity_id, 'address'
  FROM refs r JOIN (SELECT * FROM read_parquet('{s2}') UNION ALL SELECT * FROM read_parquet('{s3}')) x
    ON r.country=x.country AND r.address_key=x.address_key AND r.address_key <> ''
)
SELECT * FROM all_candidates
"""
con.execute("CREATE OR REPLACE TABLE nb04_candidates AS " + query)
stats = con.execute("SELECT COUNT(*) AS rows, COUNT(DISTINCT source1_entity_id) AS s1_with_candidates, COUNT(DISTINCT candidate_id) AS candidate_ids FROM nb04_candidates").fetchone()
print({'candidate_rows': stats[0], 's1_with_candidates': stats[1], 'candidate_ids': stats[2]})

## Holdout recall diagnostic

The ground-truth join below checks whether each labelled positive is retained by the union block. A later cell reports recall by country and cardinality only if the ground-truth file contains the corresponding fields; no submission is emitted.

In [ ]:
gt_path = DATA / 'train' / 'train_ground_truth.tsv'
gt = pd.read_csv(gt_path, sep='\t', dtype='string', keep_default_na=False)
holdout_ids = set(pd.read_parquet(holdout, columns=['entity_id'])['entity_id'])
gt = gt[gt['source1_entity_id'].isin(holdout_ids)].copy()
gt['matched_entity_ids'] = gt['matched_entity_ids'].fillna('')
positive = gt.assign(candidate_id=gt['matched_entity_ids'].str.split(',')).explode('candidate_id')
positive = positive[positive['candidate_id'].ne('')]
con.register('holdout_positive', positive[['source1_entity_id', 'candidate_id']])
recall = con.execute("""
SELECT COUNT(*) AS positives,
       SUM(CASE WHEN c.source1_entity_id IS NOT NULL THEN 1 ELSE 0 END) AS retained
FROM holdout_positive p LEFT JOIN (SELECT DISTINCT source1_entity_id, candidate_id FROM nb04_candidates) c
  ON p.source1_entity_id=c.source1_entity_id AND p.candidate_id=c.candidate_id
""").fetchone()
result = {'stage': 'S4', 'positive_pairs': int(recall[0]), 'retained_positive_pairs': int(recall[1]), 'recall': float(recall[1] / max(1, recall[0])), 'candidate_stats': {'rows': int(stats[0]), 's1_with_candidates': int(stats[1]), 'candidate_ids': int(stats[2])}}
(ART / 'nb04_recall_summary.json').write_text(json.dumps(result, indent=2) + '\n')
print(json.dumps(result, indent=2))